In [ ]:
import pandas as pd

In [ ]:
def clean_biosci_csv(input_path, output_path):
    # 1. Pull the raw text rows into a clean list
    with open(input_path, "r", encoding="utf-8", errors="ignore") as f:
        raw_lines = [line.strip() for line in f.readlines()]

    # 2. Grab the true baseline header count dynamically from row 1
    header_row = raw_lines[0]
    expected_length = len([h.strip() for h in header_row.split(",") if h.strip()])

    # 3. Track the exact Excel row numbers for any structural anomalies
    rows_with_errors = []
    current_excel_line = 1

    for line in raw_lines:
        if current_excel_line == 1 or not line:
            current_excel_line += 1
            continue

        actual_pieces = line.split(",")
        actual_length = len(actual_pieces)

        # Catch row length errors or rows ending in messy trailing commas
        if actual_length != expected_length or line.endswith(","):
            rows_with_errors.append(current_excel_line)

        current_excel_line += 1

    # Print the explicit list of broken layout rows
    if rows_with_errors:
        print(
            f"⚠️  Layout Warning: Structural issues found on Excel rows: {rows_with_errors}. Adjusting grid layout..."
        )
    else:
        print("✅ Success: All rows perfectly match the expected grid length.")

    # 4. Define the inner tool engine to force the rows to align for pandas
    def handle_bad_rows(bad_line):
        if len(bad_line) > expected_length:
            return bad_line[:expected_length]
        while len(bad_line) < expected_length:
            bad_line.append("")
        return bad_line

    # 5. Load data safely into your grid layout
    df = pd.read_csv(
        input_path, engine="python", on_bad_lines=handle_bad_rows
    )

    # 6. Loop through headers to find individual blank cells and strip spaces
    for col in df.columns:
        rows_with_missing_data = df[col].isna()
        zero_indexed_rows = df.index[rows_with_missing_data].tolist()
        one_indexed_rows = [idx + 2 for idx in zero_indexed_rows]

        if one_indexed_rows:
            print(
                f"❌ Column '{col}' has blank/missing cells on rows: {one_indexed_rows}"
            )

        df[col] = df[col].astype(str).str.strip()

    # 7. Patch missing bits with 'NA' globally and export
    df = df.fillna("NA")
    df.to_csv(output_path, index=False)
    print(f"📁 Perfect grid exported successfully to: {output_path}")

    return df